# **PRA-PEMROSESAN DATA**

In [ ]:
import pandas as pd
import numpy as np

#

In [ ]:
df_oi= pd.read_csv("/content/drive/MyDrive/Data Analyst/Project/Analysis E-Commerce/archive/order_items.csv")
df_or= pd.read_csv("/content/drive/MyDrive/Data Analyst/Project/Analysis E-Commerce/archive/order_reviews.csv")
df_p = pd.read_csv("/content/drive/MyDrive/Data Analyst/Project/Analysis E-Commerce/archive/products.csv")
df_s= pd.read_csv("/content/drive/MyDrive/Data Analyst/Project/Analysis E-Commerce/archive/sellers.csv")
df_category=pd.read_csv("/content/drive/MyDrive/Data Analyst/Project/Analysis E-Commerce/archive/product_category_name_translation.csv")

## orders_item dataset

In [ ]:
df_oi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


Fitur penting :
* seller_id
* order_id
* product_id
* price
* freight_value

In [ ]:
df_oi.describe()

,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


In [ ]:
df_oi.duplicated().sum()

np.int64(0)

In [ ]:
order_item = df_oi[
    [
        'order_id',
        'order_item_id',
        'seller_id',
        'product_id',
        'price',
        'freight_value'
    ]]

## payment dataset

In [ ]:
df_op= pd.read_csv("/content/drive/MyDrive/Data Analyst/Project/Analysis E-Commerce/archive/order_payments.csv")
df_op.info()
display(df_op)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54


Fitur penting : order_id, payment_type, payment_value



In [ ]:
df_op.duplicated().sum()

np.int64(0)

In [ ]:
payment = df_op.drop(columns=['payment_sequential','payment_installments'])

In [ ]:
cek = payment['order_id'].value_counts() > 1
cek.sum()

np.int64(2961)

Terlihat bahwa sebagaian order_id paymentnya lebih dari sekali. Maka perlu di-agregasi dulu ke order-level

In [ ]:
payment = (payment
    .groupby('order_id')
    .agg(payment_value = ('payment_value','sum'),
         payment_type =('payment_type', lambda x : x.mode().iloc[0]))
    ).reset_index()

## review dataset

In [ ]:
df_or.info()

display(df_or)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13


Fitur penting : order_id, review_score

Kenapa review_id tidak inlcude, karena pada umumnya 1 order = 1 review

In [ ]:
review = df_or.drop(columns=['review_comment_title','review_comment_message','review_creation_date','review_answer_timestamp'])

In [ ]:
review.isna().sum()

,0
review_id,0
order_id,0
review_score,0


In [ ]:
review.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   review_id     99224 non-null  object
 1   order_id      99224 non-null  object
 2   review_score  99224 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 2.3+ MB


In [ ]:
review['order_id'].nunique() > review['review_id'].nunique()

True

Terlihat order_id lebih banyak dari review_id, artinya ada order yang tidak memiliki review. Best practice tetep perlu kita agregasi ke order level

In [ ]:
review = (
    review
    .groupby('order_id')
    .agg(
        avg_review_score=('review_score', 'mean'),
        review_count=('review_id', 'count')
    )
    .reset_index()
)

## orders dataset

In [ ]:
orders = pd.read_csv("/content/drive/MyDrive/Data Analyst/Project/Analysis E-Commerce/archive/orders.csv")

In [ ]:
orders.info()

display(orders)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


In [ ]:
orders['order_status'].value_counts()

,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


Untuk analisis ini akan difokuskan untuk status delivered

In [ ]:
orders = orders[
    (orders['order_status'] == 'delivered') &
    orders['order_approved_at'].notna() &
    orders['order_delivered_carrier_date'].notna() &
    orders['order_delivered_customer_date'].notna() &
    orders['order_estimated_delivery_date'].notna()
]

In [ ]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
Index: 96455 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       96455 non-null  object
 1   customer_id                    96455 non-null  object
 2   order_status                   96455 non-null  object
 3   order_purchase_timestamp       96455 non-null  object
 4   order_approved_at              96455 non-null  object
 5   order_delivered_carrier_date   96455 non-null  object
 6   order_delivered_customer_date  96455 non-null  object
 7   order_estimated_delivery_date  96455 non-null  object
dtypes: object(8)
memory usage: 6.6+ MB


Mengganti nama dan tipe data fitur serta merekayasa beberapa fitur agar lebih mudah dianalisis

note : ts adalah timestamp

In [ ]:
#Ganti nama dan tipe data fitur
orders['purchase_ts'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['approved_ts'] = pd.to_datetime(orders['order_approved_at'])
orders['carrier_ts'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['customer_ts'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['estimated_ts'] = pd.to_datetime(orders['order_estimated_delivery_date'])

In [ ]:
#rekayasa fitur
orders['seller_processing_days'] = (orders['carrier_ts'] - orders['approved_ts']).dt.days
orders['delivery_time_days'] = (orders['customer_ts'] - orders['carrier_ts']).dt.days
orders['total_fulfillment_days'] = (orders['customer_ts'] - orders['purchase_ts']).dt.days
orders['delivery_delay_days'] = (orders['customer_ts'] - orders['estimated_ts']).dt.days
orders['delivery_delay_days'] = orders['delivery_delay_days'].apply(lambda x: 'on time' if x <= 0 else 'late')

In [ ]:
display(orders.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_ts,approved_ts,carrier_ts,customer_ts,estimated_ts,seller_processing_days,delivery_time_days,total_fulfillment_days,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2,6,8,on time
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,12,13,on time
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,9,9,on time
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,3,9,13,on time
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,1,2,on time


In [ ]:
print(len(orders['seller_processing_days'][orders['seller_processing_days']<0]))
print(len(orders['delivery_time_days'][orders['delivery_time_days']<0]))

1350
23


terlihat bahwa seller_processing_days dan delivery_time_days banyak yang nilainya aneh, atau kurang dari 0. Itu artinya perlu ditangani

In [ ]:
orders['seller_processing_days'] = (
    orders['seller_processing_days']
    .where(orders['seller_processing_days'] >= 0)
)

orders['delivery_time_days'] = (
    orders['delivery_time_days']
    .where(orders['delivery_time_days'] >= 0)
)

In [ ]:
drop_col = ['customer_id','purchase_ts', 'approved_ts', 'carrier_ts','customer_ts','estimated_ts',
            'order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
            'order_delivered_customer_date','order_estimated_delivery_date']

orders = orders.drop(drop_col,axis=1)

In [ ]:
orders = orders.dropna(
    subset=['seller_processing_days', 'delivery_time_days']
)


## product dataset

In [ ]:
df_category.info()

display(df_category)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
...,...,...
66,flores,flowers
67,artes_e_artesanato,arts_and_craftmanship
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes


In [ ]:
df_category.duplicated().sum()

np.int64(0)

product dataset

In [ ]:
df_p.info()

display(df_p)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


product category masih berbahasa brazil, maka akan diganti menggunakan bahasa inggris dengan merge category product dataset

In [ ]:
df_product = df_p.merge(df_category, on='product_category_name', how='left')

display(df_product)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares
...,...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0,furniture_decor
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0,construction_tools_lights
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0,bed_bath_table
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0,computers_accessories


fitur penting : product_id dan product_category

In [ ]:
df_product = df_product.rename(columns={'product_category_name_english': 'product_category'})

kolom_penting = ['product_id', 'product_category']

product = df_product[kolom_penting]

display(product.head())

,product_id,product_category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,art
2,96bd76ec8810374ed1b65e291975717f,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,baby
4,9dc1a7de274444849c219cff195d0b71,housewares


# **ORDER LEVEL DATASET**

Grain : 1 row = 1 order

Join dataset order, payment, dan review pada order_id.

SYARAT : Payment dan Review sudah agregaris ke order-level

Urutan order, wajib order terlebih dahulu

In [ ]:
order_level = (
    orders
    .merge(payment, on='order_id', how='left')
    .merge(review, on='order_id', how='left')
)

## **Tahap Validasi**

In [ ]:
#Cek, apakah sudah 1 row = 1 order
#Output HARUS True
order_level['order_id'].is_unique

True

In [ ]:
order_level.isna().sum()

,0
order_id,0
order_status,0
seller_processing_days,0
delivery_time_days,0
total_fulfillment_days,0
delivery_delay_days,0
payment_value,1
payment_type,1
avg_review_score,639
review_count,639


payment_value dan payment type ada missing value, boleh dihapus untuk memperbaiki akurasi GMV karena jumlahnya yang hanya 1

untuk order_id yang tidak memiliki review score maka akan diisi dengan -1. Kalau 0 bisa ambigu menjadi rating 'jelek'. Sedangkan review_count, akan dibuat 0 atau tidak punya review

In [ ]:
# Review
order_level['has_review'] = order_level['avg_review_score'].notna().astype(int)
order_level['avg_review_score'] = order_level['avg_review_score'].fillna(-1)
order_level['review_count'] = order_level['review_count'].fillna(0)

# Payment
order_level = order_level.dropna(
    subset=['payment_value', 'payment_type']
)

In [ ]:
order_level['has_review'].value_counts()

,count
has_review,
1,94442
0,639


# **ITEM LEVEL DATASET**

Grain : item level

Join dataset :
* order_item + order_level (inner, on = order_id)
* hasil + product (left, on = product_id)

Kenapa inner di join pertama?
Karena order_level sudah difilter & bersih → hanya order valid yang ikut.

In [ ]:
item_level = (
    order_item
    .merge(order_level, on='order_id', how='inner')
    .merge(product, on='product_id', how='left')
)

## **Tahap Validasi**

In [ ]:
#Output HARUS False
# Karena 1 baris = 1 item unik, yang artinya tidak boleh duplikat
item_level[['order_id', 'order_item_id']].duplicated().any()

np.False_

In [ ]:
item_level.describe()

,order_item_id,price,freight_value,seller_processing_days,delivery_time_days,total_fulfillment_days,payment_value,avg_review_score,review_count,has_review
count,108578.000000,108578.000000,108578.000000,108578.000000,108578.000000,108578.000000,108578.000000,108578.000000,108578.000000,108578.000000
mean,1.197987,120.126245,19.942341,2.396112,8.768719,12.067224,179.636289,4.042321,0.998361,0.992457
std,0.707422,182.670944,15.708252,3.527264,8.622301,9.474990,272.310342,1.411743,0.116424,0.086522
min,1.000000,0.850000,0.000000,0.000000,0.000000,0.000000,9.590000,-1.000000,0.000000,0.000000
25%,1.000000,39.900000,13.080000,0.000000,4.000000,6.000000,65.540000,4.000000,1.000000,1.000000
50%,1.000000,74.900000,16.250000,1.000000,7.000000,10.000000,114.340000,5.000000,1.000000,1.000000
75%,1.000000,134.900000,21.150000,3.000000,11.000000,15.000000,194.910000,5.000000,1.000000,1.000000
max,21.000000,6735.000000,409.680000,125.000000,205.000000,209.000000,13664.080000,5.000000,3.000000,1.000000


In [ ]:
item_level.isna().sum()

,0
order_id,0
order_item_id,0
seller_id,0
product_id,0
price,0
freight_value,0
order_status,0
seller_processing_days,0
delivery_time_days,0
total_fulfillment_days,0


Product kategori terdapat missing value, hal ini wajar, karena bisa saja, terdapat produk yang tidak terdata pada product dataset, meningat banyaknya varian produk di sebuah e-commerce.

Untuk menangani hal tersebut, akan dibuat 'uknown' untuk produk yang tidak terdata kategorinya

In [ ]:
item_level['product_category'] = (item_level['product_category'].fillna('unknown'))

In [ ]:
# Memastikan tidak ada penambahan baris secara tidak sengaja akibat join
len(item_level) <= len(order_item)

True

# **SELLER LEVEL DATASET**

Grain : 1 Baris = 1 Seller

Merupakan dataset final

KPI	yang akan dibuat :
* total_gmv --> Kontribusi pendapatan seller
* total_orders --> Aktivitas & demand
* total_items_sold --> Volume penjualan
* avg_item_price --> Positioning harga
* median_seller_processing_days --> Efisiensi seller
* median_delivery_time_days --> Kualitas pengiriman
* late_delivery_rate --> Reliability
* avg_review_score --> Kepuasan customer
* review_coverage --> Seberapa sering seller mendapat feedback

In [ ]:
seller_level =(
    item_level
    .groupby('seller_id')
    .agg(
        # Volume dan Revenue
        total_gmv = ('price','sum'),
         total_items_sold = ('order_item_id', 'count'),
         total_orders = ('order_id', 'count'),

        # Harga
        avg_item_price = ('price', 'mean'),

        # Operasional
        median_seller_processing_days = ('seller_processing_days', 'median'),
        median_delivery_time_days = ('delivery_time_days', 'median'),
        late_delivery_rate = ('delivery_delay_days', lambda x: (x == 'late').mean()),

        # Review
        avg_review_score=('avg_review_score', lambda x: x[x >= 0].mean()),
        review_coverage=('has_review', 'mean')
    )
    .reset_index()
)

In [ ]:
dominant_category = (
    item_level
    .groupby('seller_id')['product_category']
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 'unknown')
    .reset_index()
)

seller_level = seller_level.merge(
    dominant_category,
    on='seller_id',
    how='left'
)

## **Tahap Validasi**

In [ ]:
#Grain cek
seller_level['seller_id'].is_unique

True

In [ ]:
seller_level.describe()

,total_gmv,total_items_sold,total_orders,avg_item_price,median_seller_processing_days,median_delivery_time_days,late_delivery_rate,avg_review_score,review_coverage
count,2959.000000,2959.000000,2959.000000,2959.000000,2959.000000,2959.000000,2959.000000,2954.000000,2959.000000
mean,4407.930872,36.694153,36.694153,177.757246,2.361947,6.904022,0.069310,4.152059,0.992798
std,13813.893656,118.352664,118.352664,345.682115,3.751937,5.443661,0.158527,0.798356,0.050850
min,6.500000,1.000000,1.000000,6.000000,0.000000,0.000000,0.000000,1.000000,0.000000
25%,217.000000,2.000000,2.000000,52.109071,1.000000,4.500000,0.000000,3.888889,1.000000
50%,835.940000,8.000000,8.000000,95.333333,1.000000,6.000000,0.000000,4.272727,1.000000
75%,3471.000000,25.000000,25.000000,171.809113,3.000000,8.000000,0.076923,4.714286,1.000000
max,225032.830000,1942.000000,1942.000000,6735.000000,61.000000,186.000000,1.000000,5.000000,1.000000


In [ ]:
seller_level.to_csv('seller_level.csv', index=False)